[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nickmelamed/cal-nerds-research-agent/blob/main/notebooks/demo.ipynb)

# Research paper assistant: the finished agent

This agent searches arXiv for papers, reads abstracts, looks up unfamiliar
terms on Wikipedia, and answers with citations. Afterwards it checks that
every cited paper was returned by a tool during this run.

Run the cells from top to bottom. The cells with a title and hidden code
hold the tools. You can open them with the arrow on the left.

## 1. Setup

First we install the `openai` package. It is the client library, and it
also works with OpenRouter and Groq.

In [ ]:
!pip install -q openai

In [ ]:
# @title Helpers for talking to the model
"""Client setup for OpenRouter and Groq, 429 retry, and usage helpers."""

import os
import time
from getpass import getpass

import requests

MODEL = "openai/gpt-oss-120b"
MAX_429_RETRIES = 5
DEFAULT_WAIT_S = 2
MAX_WAIT_S = 30
TOKENS_HEADER = "x-ratelimit-remaining-tokens"

BASE_URLS = {
    "openrouter": "https://openrouter.ai/api/v1",
    "groq": "https://api.groq.com/openai/v1",
}
KEY_NAMES = {"openrouter": "OPENROUTER_API_KEY", "groq": "GROQ_API_KEY"}
OPENROUTER_KEY_URL = "https://openrouter.ai/api/v1/key"
GROQ_KEY_PREFIX = "gsk_"

_sleep = time.sleep
_last_headers: dict[str, str] = {}
_active: dict[str, str | None] = {"provider": None}


def _secret(name: str) -> str | None:
    key = os.environ.get(name)
    if key:
        return key
    try:
        from google.colab import userdata

        return userdata.get(name) or None
    except Exception:
        return None


def choose_provider() -> str:
    """Return ``"groq"`` when a student's own key is set, else ``"openrouter"``."""
    return "groq" if _secret(KEY_NAMES["groq"]) else "openrouter"


def get_api_key(provider: str | None = None) -> str:
    """Return the key for ``provider`` from the environment, Colab Secrets, or a prompt."""
    name = KEY_NAMES[provider or choose_provider()]
    return _secret(name) or getpass(f"{name}: ")


def make_client(provider: str | None = None):
    """Return an OpenAI-compatible client and remember which provider it uses.

    With no ``provider`` the choice follows ``choose_provider``. A key typed at
    the prompt that starts with ``gsk_`` switches to Groq.
    """
    from openai import OpenAI

    chosen = provider or choose_provider()
    key = _secret(KEY_NAMES[chosen]) or getpass(
        "Paste your API key (the workshop key, or your own Groq key): "
    )
    if provider is None and key.startswith(GROQ_KEY_PREFIX):
        chosen = "groq"
    _active["provider"] = chosen
    return OpenAI(base_url=BASE_URLS[chosen], api_key=key)


def active_provider() -> str | None:
    """Return the provider of the last client made, for the smoke test."""
    return _active["provider"]


def _retry_wait(error: Exception, attempt: int = 0) -> float:
    headers = getattr(getattr(error, "response", None), "headers", None) or {}
    try:
        wait = float(headers["retry-after"])
    except (KeyError, TypeError, ValueError):
        wait = DEFAULT_WAIT_S * 2**attempt
    return min(max(wait, 1), MAX_WAIT_S)


def chat(client, **kwargs):
    """Call the chat endpoint, retrying on 429.

    Response headers are kept for ``remaining_tokens``. After the retries run
    out the last 429 error is raised.
    """
    raw_api = client.chat.completions.with_raw_response
    for attempt in range(MAX_429_RETRIES + 1):
        try:
            raw = raw_api.create(**kwargs)
        except Exception as error:
            if getattr(error, "status_code", None) != 429 or attempt == MAX_429_RETRIES:
                raise
            wait = _retry_wait(error, attempt)
            print(f"The shared key is busy, retrying in {wait:g}s")
            _sleep(wait)
        else:
            _last_headers.clear()
            _last_headers.update({k.lower(): v for k, v in raw.headers.items()})
            return raw.parse()


def remaining_tokens() -> int | None:
    """Return the tokens left this minute from the last response, if reported.

    Only Groq sends this header.
    """
    try:
        return int(_last_headers[TOKENS_HEADER])
    except (KeyError, ValueError):
        return None


def openrouter_credit(key: str | None = None) -> float | None:
    """Return the credit left on the OpenRouter key in dollars, or None.

    None means the key has no limit set or the request failed.
    """
    key = key or _secret(KEY_NAMES["openrouter"])
    try:
        reply = requests.get(
            OPENROUTER_KEY_URL, headers={"Authorization": f"Bearer {key}"}, timeout=10
        )
        reply.raise_for_status()
        remaining = reply.json()["data"]["limit_remaining"]
    except Exception:
        return None
    return float(remaining) if remaining is not None else None


def smoke_test(client, model: str = MODEL) -> bool:
    """Send one tiny request and print whether the key and model work."""
    try:
        chat(client, model=model, messages=[{"role": "user", "content": "Say ready."}])
    except Exception as error:
        print(f"Smoke test failed with {active_provider()} ({type(error).__name__}).")
        print("Check that your key is right and has credit, then run this cell again.")
        return False
    print(f"Smoke test passed. Using {active_provider()} with model {model}.")
    return True

## 2. The tools

A tool is a Python function the model can ask us to run. There are three.
`search_arxiv` finds papers, `get_abstract` reads one abstract, and
`lookup_wikipedia` explains a term.

In [ ]:
# @title Tool helpers and settings
"""Tools the research agent calls: arXiv search, arXiv abstracts, Wikipedia."""

import json
import re
import time
import xml.etree.ElementTree as ET
from urllib.parse import quote, urlencode

import requests

ARXIV_URL = "https://export.arxiv.org/api/query"
WIKI_SUMMARY_URL = "https://en.wikipedia.org/api/rest_v1/page/summary/"
WIKI_SEARCH_URL = "https://en.wikipedia.org/w/api.php"
USER_AGENT = "cal-nerds-research-agent/0.1 (educational workshop notebook)"

ARXIV_DELAY_S = 3
MAX_RESULTS = 3
ABSTRACT_WORDS = 150
WIKI_SENTENCES = 3
MAX_RETRIES = 3
BACKOFF_S = 2
TIMEOUT_S = 20

ATOM = {"a": "http://www.w3.org/2005/Atom"}

# Raw response bodies served in OFFLINE mode, keyed by _offline_key.
OFFLINE = False
OFFLINE_DATA: dict[str, str] = {}

# Swapped out in tests so the delay logic runs without real waiting.
_clock = time.monotonic
_sleep = time.sleep

_last_arxiv_call: float | None = None
_search_cache: dict[tuple[str, int], list[dict] | str] = {}
_papers: dict[str, dict] = {}


def reset_cache() -> None:
    """Forget cached searches and papers, for the start of a new agent run."""
    _search_cache.clear()
    _papers.clear()


def _offline_key(url: str, params: dict | None) -> str:
    return url + ("?" + urlencode(sorted(params.items())) if params else "")


def _wait_for_arxiv() -> None:
    global _last_arxiv_call
    if _last_arxiv_call is not None:
        remaining = ARXIV_DELAY_S - (_clock() - _last_arxiv_call)
        if remaining > 0:
            _sleep(remaining)
    _last_arxiv_call = _clock()


def _arxiv_get(params: dict) -> str | None:
    """Return the Atom text, or None after retries."""
    if OFFLINE:
        return OFFLINE_DATA.get(_offline_key(ARXIV_URL, params))
    for attempt in range(MAX_RETRIES):
        _wait_for_arxiv()
        try:
            resp = requests.get(
                ARXIV_URL, params=params, headers={"User-Agent": USER_AGENT}, timeout=TIMEOUT_S
            )
        except requests.RequestException:
            resp = None
        if resp is not None and resp.status_code == 200:
            return resp.text
        if resp is not None and resp.status_code not in (429, 503):
            return None
        _sleep(BACKOFF_S * 2**attempt)
    return None


def _wiki_get(url: str, params: dict | None = None) -> requests.Response | str | None:
    """Fetch a Wikipedia URL. Offline mode returns the stored body as text."""
    if OFFLINE:
        return OFFLINE_DATA.get(_offline_key(url, params))
    try:
        return requests.get(
            url, params=params, headers={"User-Agent": USER_AGENT}, timeout=TIMEOUT_S
        )
    except requests.RequestException:
        return None


def _clean(text: str) -> str:
    return " ".join(text.split())


def _truncate_words(text: str, limit: int = ABSTRACT_WORDS) -> str:
    words = text.split()
    if len(words) <= limit:
        return " ".join(words)
    return " ".join(words[:limit]) + " ..."


def _truncate_sentences(text: str, limit: int = WIKI_SENTENCES) -> str:
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    return " ".join(sentences[:limit])


def _normalize_id(paper_id: str) -> str:
    pid = paper_id.strip()
    pid = re.sub(r"^(https?://)?(www\.)?arxiv\.org/(abs|pdf)/", "", pid)
    pid = re.sub(r"^arxiv:", "", pid, flags=re.IGNORECASE)
    pid = re.sub(r"\.pdf$", "", pid)
    return re.sub(r"v\d+$", "", pid)


def _parse_entries(xml_text: str) -> list[dict]:
    root = ET.fromstring(xml_text)
    papers = []
    for entry in root.findall("a:entry", ATOM):
        raw_id = entry.findtext("a:id", default="", namespaces=ATOM)
        title = _clean(entry.findtext("a:title", default="", namespaces=ATOM))
        # arXiv answers a bad id_list with a single error entry, not an HTTP error.
        if not title or title.lower() == "error" or "/api/errors" in raw_id:
            continue
        paper_id = _normalize_id(raw_id)
        authors = [
            _clean(a.findtext("a:name", default="", namespaces=ATOM))
            for a in entry.findall("a:author", ATOM)
        ]
        papers.append(
            {
                "id": paper_id,
                "title": title,
                "authors": authors[:3] + (["et al."] if len(authors) > 3 else []),
                "date": entry.findtext("a:published", default="", namespaces=ATOM)[:10],
                "link": f"https://arxiv.org/abs/{paper_id}",
                "abstract": _truncate_words(
                    entry.findtext("a:summary", default="", namespaces=ATOM)
                ),
            }
        )
    return papers






def _wiki_summary(title: str) -> dict | None:
    resp = _wiki_get(WIKI_SUMMARY_URL + quote(title.replace(" ", "_"), safe=""))
    if resp is None:
        return None
    if isinstance(resp, str):
        data = _loads(resp)
    elif resp.status_code == 200:
        data = _loads(resp.text)
    else:
        return None
    if not data or data.get("type") == "disambiguation" or not data.get("extract"):
        return None
    return {
        "title": data.get("title", title),
        "summary": _truncate_sentences(data["extract"]),
        "url": data.get("content_urls", {}).get("desktop", {}).get("page", ""),
    }


def _loads(text: str) -> dict | None:
    try:
        data = json.loads(text)
    except ValueError:
        return None
    return data if isinstance(data, dict) else None

In [ ]:
# @title search_arxiv: find papers
def search_arxiv(query: str, max_results: int = MAX_RESULTS) -> list[dict] | str:
    """Search arXiv and return id, title, authors, date, and link per paper.

    Returns a message string when nothing is found or arXiv is unreachable.
    Abstracts are cached for ``get_abstract``.
    """
    query = (query or "").strip()
    if not query:
        return "The search query was empty. Give a few keywords describing the topic."
    max_results = max(1, min(int(max_results), MAX_RESULTS))
    key = (query.lower(), max_results)
    if key in _search_cache:
        return _search_cache[key]

    params = {
        "search_query": f"all:{query}",
        "start": 0,
        "max_results": max_results,
        "sortBy": "relevance",
    }
    text = _arxiv_get(params)
    if text is None:
        return (
            "arXiv could not be reached. Try again, or use lookup_wikipedia "
            "for background on the topic."
        )
    try:
        papers = _parse_entries(text)
    except ET.ParseError:
        return "arXiv returned a response that could not be read. Try a different query."
    if not papers:
        return f"No papers found for '{query}'. Try broader or different keywords."

    for paper in papers:
        _papers[paper["id"]] = paper
    result = [{k: v for k, v in p.items() if k != "abstract"} for p in papers]
    _search_cache[key] = result
    return result

In [ ]:
# @title get_abstract: read one abstract
def get_abstract(paper_id: str) -> dict | str:
    """Return the id, title, and abstract (about 150 words) for one arXiv paper."""
    pid = _normalize_id(paper_id or "")
    if not pid:
        return "No paper id was given. Use an id from search_arxiv, like 2101.00001."
    if pid in _papers:
        paper = _papers[pid]
        return {"id": pid, "title": paper["title"], "abstract": paper["abstract"]}

    text = _arxiv_get({"id_list": pid, "max_results": 1})
    if text is None:
        return "arXiv could not be reached. Try again in a moment."
    try:
        papers = _parse_entries(text)
    except ET.ParseError:
        return "arXiv returned a response that could not be read."
    if not papers:
        return f"No paper found with id '{pid}'. Check the id from search_arxiv."
    paper = papers[0]
    _papers[paper["id"]] = paper
    return {"id": paper["id"], "title": paper["title"], "abstract": paper["abstract"]}

In [ ]:
# @title lookup_wikipedia: explain a term
def lookup_wikipedia(term: str) -> dict | str:
    """Return a short Wikipedia summary, searching if the exact title misses."""
    term = (term or "").strip()
    if not term:
        return "The term was empty. Give a word or short phrase to look up."

    found = _wiki_summary(term)
    if found:
        return found

    params = {
        "action": "query",
        "list": "search",
        "srsearch": term,
        "srlimit": 1,
        "format": "json",
    }
    resp = _wiki_get(WIKI_SEARCH_URL, params)
    body = resp if isinstance(resp, str) else (resp.text if resp is not None else None)
    data = _loads(body) if body else None
    hits = (data or {}).get("query", {}).get("search", [])
    if hits:
        found = _wiki_summary(hits[0]["title"])
        if found:
            return found
    return f"No Wikipedia page found for '{term}'. Try a more common name for the concept."

## 3. Connect to the model

`OFFLINE = False` means the tools call arXiv and Wikipedia. Set it to
`True` to use saved responses instead. It only replaces the tools. The
model still needs a working key. Saved responses cover only a few searches,
so most questions get a "could not be reached" message in this mode.

If you have your own Groq key in Colab Secrets as `GROQ_API_KEY`, the
notebook uses it. Otherwise it uses `OPENROUTER_API_KEY`, and if neither is
set it asks you to paste a key.

In [ ]:
# @title Saved responses for OFFLINE mode
OFFLINE_DATA.update(json.loads("{\"https://en.wikipedia.org/api/rest_v1/page/summary/AlphaFold\": \"{\\\"type\\\":\\\"standard\\\",\\\"title\\\":\\\"AlphaFold\\\",\\\"displaytitle\\\":\\\"<span lang=\\\\\\\"en\\\\\\\" dir=\\\\\\\"ltr\\\\\\\"><span class=\\\\\\\"mw-page-title-main\\\\\\\">AlphaFold</span></span>\\\",\\\"namespace\\\":{\\\"id\\\":0,\\\"text\\\":\\\"\\\"},\\\"wikibase_item\\\":\\\"Q60827595\\\",\\\"titles\\\":{\\\"canonical\\\":\\\"AlphaFold\\\",\\\"normalized\\\":\\\"AlphaFold\\\",\\\"display\\\":\\\"<span lang=\\\\\\\"en\\\\\\\" dir=\\\\\\\"ltr\\\\\\\"><span class=\\\\\\\"mw-page-title-main\\\\\\\">AlphaFold</span></span>\\\"},\\\"pageid\\\":59766171,\\\"thumbnail\\\":{\\\"source\\\":\\\"https://thumb.wikimedia.org/wikipedia/commons/thumb/e/ee/T1044-alphafold3-layey-by-layer.gif/330px-T1044-alphafold3-layey-by-layer.gif?utm_source=en.wikipedia.org&utm_campaign=api&utm_content=thumbnail\\\",\\\"width\\\":330,\\\"height\\\":240},\\\"originalimage\\\":{\\\"source\\\":\\\"https://upload.wikimedia.org/wikipedia/commons/e/ee/T1044-alphafold3-layey-by-layer.gif?utm_source=en.wikipedia.org&utm_campaign=api&utm_content=thumbnail_unscaled\\\",\\\"width\\\":496,\\\"height\\\":360},\\\"lang\\\":\\\"en\\\",\\\"dir\\\":\\\"ltr\\\",\\\"revision\\\":\\\"1385537572\\\",\\\"tid\\\":\\\"4ece45a2-dfa3-11f1-86f5-6504c53178c1\\\",\\\"timestamp\\\":\\\"2026-11-13T20:46:32Z\\\",\\\"description\\\":\\\"Artificial intelligence program by DeepMind\\\",\\\"description_source\\\":\\\"local\\\",\\\"content_urls\\\":{\\\"desktop\\\":{\\\"page\\\":\\\"https://en.wikipedia.org/wiki/AlphaFold\\\",\\\"revisions\\\":\\\"https://en.wikipedia.org/wiki/AlphaFold?action=history\\\",\\\"edit\\\":\\\"https://en.wikipedia.org/wiki/AlphaFold?action=edit\\\",\\\"talk\\\":\\\"https://en.wikipedia.org/wiki/Talk:AlphaFold\\\"},\\\"mobile\\\":{\\\"page\\\":\\\"https://en.wikipedia.org/wiki/AlphaFold\\\",\\\"revisions\\\":\\\"https://en.wikipedia.org/wiki/Special:History/AlphaFold\\\",\\\"edit\\\":\\\"https://en.wikipedia.org/wiki/AlphaFold?action=edit\\\",\\\"talk\\\":\\\"https://en.wikipedia.org/wiki/Talk:AlphaFold\\\"}},\\\"extract\\\":\\\"AlphaFold is an artificial intelligence (AI) program developed by DeepMind, a subsidiary of Alphabet, which performs predictions of protein structure. It is designed using deep learning techniques.\\\",\\\"extract_html\\\":\\\"<p><b>AlphaFold</b> is an artificial intelligence (AI) program developed by DeepMind, a subsidiary of Alphabet, which performs predictions of protein structure. It is designed using deep learning techniques.</p>\\\"}\", \"https://en.wikipedia.org/api/rest_v1/page/summary/Attention_%28machine_learning%29\": \"{\\\"type\\\":\\\"standard\\\",\\\"title\\\":\\\"Attention (machine learning)\\\",\\\"displaytitle\\\":\\\"<span lang=\\\\\\\"en\\\\\\\" dir=\\\\\\\"ltr\\\\\\\"><span class=\\\\\\\"mw-page-title-main\\\\\\\">Attention (machine learning)</span></span>\\\",\\\"namespace\\\":{\\\"id\\\":0,\\\"text\\\":\\\"\\\"},\\\"wikibase_item\\\":\\\"Q103701642\\\",\\\"titles\\\":{\\\"canonical\\\":\\\"Attention_(machine_learning)\\\",\\\"normalized\\\":\\\"Attention (machine learning)\\\",\\\"display\\\":\\\"<span lang=\\\\\\\"en\\\\\\\" dir=\\\\\\\"ltr\\\\\\\"><span class=\\\\\\\"mw-page-title-main\\\\\\\">Attention (machine learning)</span></span>\\\"},\\\"pageid\\\":66001552,\\\"thumbnail\\\":{\\\"source\\\":\\\"https://thumb.wikimedia.org/wikipedia/commons/thumb/6/62/Attention_mechanism_overview.svg/330px-Attention_mechanism_overview.svg.png?utm_source=en.wikipedia.org&utm_campaign=api&utm_content=thumbnail\\\",\\\"width\\\":330,\\\"height\\\":246},\\\"originalimage\\\":{\\\"source\\\":\\\"https://thumb.wikimedia.org/wikipedia/commons/thumb/6/62/Attention_mechanism_overview.svg/330px-Attention_mechanism_overview.svg.png?utm_source=en.wikipedia.org&utm_campaign=api&utm_content=thumbnail\\\",\\\"width\\\":251,\\\"height\\\":187},\\\"lang\\\":\\\"en\\\",\\\"dir\\\":\\\"ltr\\\",\\\"revision\\\":\\\"1386512027\\\",\\\"tid\\\":\\\"c5a27ef0-e3a7-11f1-90d7-6fc120768809\\\",\\\"timestamp\\\":\\\"2026-11-18T23:28:34Z\\\",\\\"description\\\":\\\"Machine learning technique\\\",\\\"description_source\\\":\\\"local\\\",\\\"content_urls\\\":{\\\"desktop\\\":{\\\"page\\\":\\\"https://en.wikipedia.org/wiki/Attention_(machine_learning)\\\",\\\"revisions\\\":\\\"https://en.wikipedia.org/wiki/Attention_(machine_learning)?action=history\\\",\\\"edit\\\":\\\"https://en.wikipedia.org/wiki/Attention_(machine_learning)?action=edit\\\",\\\"talk\\\":\\\"https://en.wikipedia.org/wiki/Talk:Attention_(machine_learning)\\\"},\\\"mobile\\\":{\\\"page\\\":\\\"https://en.wikipedia.org/wiki/Attention_(machine_learning)\\\",\\\"revisions\\\":\\\"https://en.wikipedia.org/wiki/Special:History/Attention_(machine_learning)\\\",\\\"edit\\\":\\\"https://en.wikipedia.org/wiki/Attention_(machine_learning)?action=edit\\\",\\\"talk\\\":\\\"https://en.wikipedia.org/wiki/Talk:Attention_(machine_learning)\\\"}},\\\"extract\\\":\\\"In machine learning, attention is a method that determines the importance of, or \\\\\\\"attends to\\\\\\\", each component in a sequence relative to the other components in that sequence. In natural language processing, importance is represented by weights assigned to each word in a sentence. More generally, attention encodes vectors called token embeddings across a fixed-width sequence that can range from tens to millions of tokens in size.\\\",\\\"extract_html\\\":\\\"<p>In machine learning, <b>attention</b> is a method that determines the importance of, or \\\\\\\"attends to\\\\\\\", each component in a sequence relative to the other components in that sequence. In natural language processing, importance is represented by weights assigned to each word in a sentence. More generally, attention encodes vectors called token embeddings across a fixed-width sequence that can range from tens to millions of tokens in size.</p>\\\"}\", \"https://en.wikipedia.org/api/rest_v1/page/summary/Protein_folding\": \"{\\\"type\\\":\\\"standard\\\",\\\"title\\\":\\\"Protein folding\\\",\\\"displaytitle\\\":\\\"<span lang=\\\\\\\"en\\\\\\\" dir=\\\\\\\"ltr\\\\\\\"><span class=\\\\\\\"mw-page-title-main\\\\\\\">Protein folding</span></span>\\\",\\\"namespace\\\":{\\\"id\\\":0,\\\"text\\\":\\\"\\\"},\\\"wikibase_item\\\":\\\"Q847556\\\",\\\"titles\\\":{\\\"canonical\\\":\\\"Protein_folding\\\",\\\"normalized\\\":\\\"Protein folding\\\",\\\"display\\\":\\\"<span lang=\\\\\\\"en\\\\\\\" dir=\\\\\\\"ltr\\\\\\\"><span class=\\\\\\\"mw-page-title-main\\\\\\\">Protein folding</span></span>\\\"},\\\"pageid\\\":52085,\\\"thumbnail\\\":{\\\"source\\\":\\\"https://thumb.wikimedia.org/wikipedia/commons/thumb/0/05/Protein_structure.png/330px-Protein_structure.png?utm_source=en.wikipedia.org&utm_campaign=api&utm_content=thumbnail\\\",\\\"width\\\":330,\\\"height\\\":248},\\\"originalimage\\\":{\\\"source\\\":\\\"https://upload.wikimedia.org/wikipedia/commons/0/05/Protein_structure.png?utm_source=en.wikipedia.org&utm_campaign=api&utm_content=thumbnail_unscaled\\\",\\\"width\\\":1344,\\\"height\\\":1008},\\\"lang\\\":\\\"en\\\",\\\"dir\\\":\\\"ltr\\\",\\\"revision\\\":\\\"1385704802\\\",\\\"tid\\\":\\\"6e7e6e9f-e06e-11f1-b343-f459782847ef\\\",\\\"timestamp\\\":\\\"2026-11-14T21:00:33Z\\\",\\\"description\\\":\\\"Change of a linear protein chain to a 3D structure\\\",\\\"description_source\\\":\\\"local\\\",\\\"content_urls\\\":{\\\"desktop\\\":{\\\"page\\\":\\\"https://en.wikipedia.org/wiki/Protein_folding\\\",\\\"revisions\\\":\\\"https://en.wikipedia.org/wiki/Protein_folding?action=history\\\",\\\"edit\\\":\\\"https://en.wikipedia.org/wiki/Protein_folding?action=edit\\\",\\\"talk\\\":\\\"https://en.wikipedia.org/wiki/Talk:Protein_folding\\\"},\\\"mobile\\\":{\\\"page\\\":\\\"https://en.wikipedia.org/wiki/Protein_folding\\\",\\\"revisions\\\":\\\"https://en.wikipedia.org/wiki/Special:History/Protein_folding\\\",\\\"edit\\\":\\\"https://en.wikipedia.org/wiki/Protein_folding?action=edit\\\",\\\"talk\\\":\\\"https://en.wikipedia.org/wiki/Talk:Protein_folding\\\"}},\\\"extract\\\":\\\"Protein folding is the physical process by which a protein, after synthesis by a ribosome as a linear chain of amino acids, changes from an unstable random coil into a more ordered three-dimensional structure. This structure permits the protein to become biologically functional or active.\\\",\\\"extract_html\\\":\\\"<p><b>Protein folding</b> is the physical process by which a protein, after synthesis by a ribosome as a linear chain of amino acids, changes from an unstable random coil into a more ordered three-dimensional structure. This structure permits the protein to become biologically functional or active.</p>\\\"}\", \"https://en.wikipedia.org/api/rest_v1/page/summary/how_proteins_fold\": \"{\\\"status\\\":404,\\\"type\\\":\\\"Internal error\\\"}\", \"https://en.wikipedia.org/api/rest_v1/page/summary/protein_folding_problem\": \"{\\\"type\\\":\\\"standard\\\",\\\"title\\\":\\\"Protein structure prediction\\\",\\\"displaytitle\\\":\\\"<span lang=\\\\\\\"en\\\\\\\" dir=\\\\\\\"ltr\\\\\\\"><span class=\\\\\\\"mw-page-title-main\\\\\\\">Protein structure prediction</span></span>\\\",\\\"namespace\\\":{\\\"id\\\":0,\\\"text\\\":\\\"\\\"},\\\"wikibase_item\\\":\\\"Q899656\\\",\\\"titles\\\":{\\\"canonical\\\":\\\"Protein_structure_prediction\\\",\\\"normalized\\\":\\\"Protein structure prediction\\\",\\\"display\\\":\\\"<span lang=\\\\\\\"en\\\\\\\" dir=\\\\\\\"ltr\\\\\\\"><span class=\\\\\\\"mw-page-title-main\\\\\\\">Protein structure prediction</span></span>\\\"},\\\"pageid\\\":306769,\\\"thumbnail\\\":{\\\"source\\\":\\\"https://thumb.wikimedia.org/wikipedia/commons/thumb/a/a6/Protein-structure.png/330px-Protein-structure.png?utm_source=en.wikipedia.org&utm_campaign=api&utm_content=thumbnail\\\",\\\"width\\\":330,\\\"height\\\":450},\\\"originalimage\\\":{\\\"source\\\":\\\"https://upload.wikimedia.org/wikipedia/commons/a/a6/Protein-structure.png?utm_source=en.wikipedia.org&utm_campaign=api&utm_content=thumbnail_unscaled\\\",\\\"width\\\":396,\\\"height\\\":540},\\\"lang\\\":\\\"en\\\",\\\"dir\\\":\\\"ltr\\\",\\\"revision\\\":\\\"1383869780\\\",\\\"tid\\\":\\\"72c8c1d9-d8e7-11f1-aa14-2451c8e65677\\\",\\\"timestamp\\\":\\\"2026-11-05T07:06:40Z\\\",\\\"description\\\":\\\"Type of biological prediction\\\",\\\"description_source\\\":\\\"local\\\",\\\"content_urls\\\":{\\\"desktop\\\":{\\\"page\\\":\\\"https://en.wikipedia.org/wiki/Protein_structure_prediction\\\",\\\"revisions\\\":\\\"https://en.wikipedia.org/wiki/Protein_structure_prediction?action=history\\\",\\\"edit\\\":\\\"https://en.wikipedia.org/wiki/Protein_structure_prediction?action=edit\\\",\\\"talk\\\":\\\"https://en.wikipedia.org/wiki/Talk:Protein_structure_prediction\\\"},\\\"mobile\\\":{\\\"page\\\":\\\"https://en.wikipedia.org/wiki/Protein_structure_prediction\\\",\\\"revisions\\\":\\\"https://en.wikipedia.org/wiki/Special:History/Protein_structure_prediction\\\",\\\"edit\\\":\\\"https://en.wikipedia.org/wiki/Protein_structure_prediction?action=edit\\\",\\\"talk\\\":\\\"https://en.wikipedia.org/wiki/Talk:Protein_structure_prediction\\\"}},\\\"extract\\\":\\\"Protein structure prediction is the inference of the three-dimensional structure of a protein from its amino acid sequence\\u2014that is, the prediction of its secondary and tertiary structure from primary structure. Structure prediction is different from the inverse problem of protein design.\\\",\\\"extract_html\\\":\\\"<p><b>Protein structure prediction</b> is the inference of the three-dimensional structure of a protein from its amino acid sequence\\u2014that is, the prediction of its secondary and tertiary structure from primary structure. Structure prediction is different from the inverse problem of protein design.</p>\\\"}\", \"https://en.wikipedia.org/api/rest_v1/page/summary/qqqzzzxxkk\": \"{\\\"status\\\":404,\\\"type\\\":\\\"Internal error\\\"}\", \"https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srlimit=1&srsearch=how+proteins+fold\": \"{\\\"batchcomplete\\\":\\\"\\\",\\\"continue\\\":{\\\"sroffset\\\":1,\\\"continue\\\":\\\"-||\\\"},\\\"query\\\":{\\\"searchinfo\\\":{\\\"totalhits\\\":3044},\\\"search\\\":[{\\\"ns\\\":0,\\\"title\\\":\\\"Protein folding\\\",\\\"pageid\\\":52085,\\\"size\\\":78236,\\\"wordcount\\\":8781,\\\"snippet\\\":\\\"misfolded <span class=\\\\\\\"searchmatch\\\\\\\">proteins</span>, the infectious varieties of which are known as prions. Many allergies are caused by the incorrect <span class=\\\\\\\"searchmatch\\\\\\\">folding</span> of some <span class=\\\\\\\"searchmatch\\\\\\\">proteins</span> because the\\\",\\\"timestamp\\\":\\\"2026-11-14T21:00:33Z\\\"}]}}\", \"https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srlimit=1&srsearch=qqqzzzxxkk\": \"{\\\"batchcomplete\\\":\\\"\\\",\\\"query\\\":{\\\"searchinfo\\\":{\\\"totalhits\\\":0},\\\"search\\\":[]}}\", \"https://export.arxiv.org/api/query?id_list=0000.00000&max_results=1\": \"<?xml version='1.0' encoding='UTF-8'?>\\n<feed xmlns:opensearch=\\\"http://a9.com/-/spec/opensearch/1.1/\\\" xmlns:arxiv=\\\"http://arxiv.org/schemas/atom\\\" xmlns=\\\"http://www.w3.org/2005/Atom\\\">\\n  <id>https://arxiv.org/api/pogpZ7ehwmNW3Cwu7NT1Bp1f/zY</id>\\n  <title>arXiv Query: search_query=&amp;id_list=0000.00000&amp;start=0&amp;max_results=1</title>\\n  <updated>2026-11-21T18:30:25Z</updated>\\n  <link href=\\\"https://arxiv.org/api/query?search_query=&amp;start=0&amp;max_results=1&amp;id_list=0000.00000\\\" type=\\\"application/atom+xml\\\"/>\\n  <opensearch:itemsPerPage>1</opensearch:itemsPerPage>\\n  <opensearch:totalResults>0</opensearch:totalResults>\\n  <opensearch:startIndex>0</opensearch:startIndex>\\n</feed>\\n\", \"https://export.arxiv.org/api/query?max_results=3&search_query=all%3Alarge+language+model+agents+tool+use&sortBy=relevance&start=0\": \"<?xml version='1.0' encoding='UTF-8'?>\\n<feed xmlns:opensearch=\\\"http://a9.com/-/spec/opensearch/1.1/\\\" xmlns:arxiv=\\\"http://arxiv.org/schemas/atom\\\" xmlns=\\\"http://www.w3.org/2005/Atom\\\">\\n  <id>https://arxiv.org/api/Tpv7xMinTTR/1cgPMSz5O+pM5Tw</id>\\n  <title>arXiv Query: search_query=all:large OR all:language OR all:model OR all:agents OR all:tool OR all:use&amp;id_list=&amp;start=0&amp;max_results=3</title>\\n  <updated>2026-11-21T18:42:19Z</updated>\\n  <link href=\\\"https://arxiv.org/api/query?search_query=all:large+OR+(all:language+OR+(all:model+OR+(all:agents+OR+(all:tool+OR+all:use))))&amp;start=0&amp;max_results=3&amp;id_list=\\\" type=\\\"application/atom+xml\\\"/>\\n  <opensearch:itemsPerPage>3</opensearch:itemsPerPage>\\n  <opensearch:totalResults>2292854</opensearch:totalResults>\\n  <opensearch:startIndex>0</opensearch:startIndex>\\n  <entry>\\n    <id>http://arxiv.org/abs/2402.11651v2</id>\\n    <title>Learning From Failure: Integrating Negative Examples when Fine-tuning Large Language Models as Agents</title>\\n    <updated>2024-04-16T11:41:13Z</updated>\\n    <link href=\\\"https://arxiv.org/abs/2402.11651v2\\\" rel=\\\"alternate\\\" type=\\\"text/html\\\"/>\\n    <link href=\\\"https://arxiv.org/pdf/2402.11651v2\\\" rel=\\\"related\\\" type=\\\"application/pdf\\\" title=\\\"pdf\\\"/>\\n    <summary>Large language models (LLMs) have achieved success in acting as agents, which interact with environments through tools such as search engines. However, LLMs are optimized for language generation instead of tool use during training or alignment, limiting their effectiveness as agents. To resolve this problem, previous work has first collected interaction trajectories between LLMs and environments, using only trajectories that successfully finished the task to fine-tune smaller models, making fine-tuning data scarce and acquiring it both difficult and costly. Discarding failed trajectories also leads to significant wastage of data and resources and limits the possible optimization paths during fine-tuning. In this paper, we argue that unsuccessful trajectories offer valuable insights, and LLMs can learn from these trajectories through appropriate quality control and fine-tuning strategies. By simply adding a prefix or suffix that tells the model whether to generate a successful trajectory during training, we improve model performance by a large margin on mathematical reasoning, multi-hop question answering, and strategic question answering tasks. We further analyze the inference results and find that our method provides a better trade-off between valuable information and errors in unsuccessful trajectories. To our knowledge, we are the first to demonstrate the value of negative trajectories and their application in agent-tunning scenarios. Our findings offer guidance for developing better agent-tuning methods and low-resource data usage techniques.</summary>\\n    <category term=\\\"cs.CL\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <published>2024-02-18T17:10:07Z</published>\\n    <arxiv:comment>Agent, LLM, Large Language Model</arxiv:comment>\\n    <arxiv:primary_category term=\\\"cs.CL\\\"/>\\n    <author>\\n      <name>Renxi Wang</name>\\n    </author>\\n    <author>\\n      <name>Haonan Li</name>\\n    </author>\\n    <author>\\n      <name>Xudong Han</name>\\n    </author>\\n    <author>\\n      <name>Yixuan Zhang</name>\\n    </author>\\n    <author>\\n      <name>Timothy Baldwin</name>\\n    </author>\\n  </entry>\\n  <entry>\\n    <id>http://arxiv.org/abs/2404.01023v1</id>\\n    <title>Large Language Model Evaluation Via Multi AI Agents: Preliminary results</title>\\n    <updated>2024-04-01T10:06:04Z</updated>\\n    <link href=\\\"https://arxiv.org/abs/2404.01023v1\\\" rel=\\\"alternate\\\" type=\\\"text/html\\\"/>\\n    <link href=\\\"https://arxiv.org/pdf/2404.01023v1\\\" rel=\\\"related\\\" type=\\\"application/pdf\\\" title=\\\"pdf\\\"/>\\n    <summary>As Large Language Models (LLMs) have become integral to both research and daily operations, rigorous evaluation is crucial. This assessment is important not only for individual tasks but also for understanding their societal impact and potential risks. Despite extensive efforts to examine LLMs from various perspectives, there is a noticeable lack of multi-agent AI models specifically designed to evaluate the performance of different LLMs. To address this gap, we introduce a novel multi-agent AI model that aims to assess and compare the performance of various LLMs. Our model consists of eight distinct AI agents, each responsible for retrieving code based on a common description from different advanced language models, including GPT-3.5, GPT-3.5 Turbo, GPT-4, GPT-4 Turbo, Google Bard, LLAMA, and Hugging Face. Our developed model utilizes the API of each language model to retrieve code for a given high-level description. Additionally, we developed a verification agent, tasked with the critical role of evaluating the code generated by its counterparts. We integrate the HumanEval benchmark into our verification agent to assess the generated code's performance, providing insights into their respective capabilities and efficiencies. Our initial results indicate that the GPT-3.5 Turbo model's performance is comparatively better than the other models. This preliminary analysis serves as a benchmark, comparing their performances side by side. Our future goal is to enhance the evaluation process by incorporating the Massively Multitask Benchmark for Python (MBPP) benchmark, which is expected to further refine our assessment. Additionally, we plan to share our developed model with twenty practitioners from various backgrounds to test our model and collect their feedback for further improvement.</summary>\\n    <category term=\\\"cs.SE\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <published>2024-04-01T10:06:04Z</published>\\n    <arxiv:comment>10 pages, 1 figure</arxiv:comment>\\n    <arxiv:primary_category term=\\\"cs.SE\\\"/>\\n    <arxiv:journal_ref>ICLR 2024 Workshop on Large Language Model (LLM) Agents, 2024</arxiv:journal_ref>\\n    <author>\\n      <name>Zeeshan Rasheed</name>\\n    </author>\\n    <author>\\n      <name>Muhammad Waseem</name>\\n    </author>\\n    <author>\\n      <name>Kari Syst\\u00e4</name>\\n    </author>\\n    <author>\\n      <name>Pekka Abrahamsson</name>\\n    </author>\\n  </entry>\\n  <entry>\\n    <id>http://arxiv.org/abs/2111.01762v1</id>\\n    <title>AIRCC-Clim: a user-friendly tool for generating regional probabilistic climate change scenarios and risk measures</title>\\n    <updated>2021-10-30T11:09:03Z</updated>\\n    <link href=\\\"https://arxiv.org/abs/2111.01762v1\\\" rel=\\\"alternate\\\" type=\\\"text/html\\\"/>\\n    <link href=\\\"https://arxiv.org/pdf/2111.01762v1\\\" rel=\\\"related\\\" type=\\\"application/pdf\\\" title=\\\"pdf\\\"/>\\n    <summary>Complex physical models are the most advanced tools available for producing realistic simulations of the climate system. However, such levels of realism imply high computational cost and restrictions on their use for policymaking and risk assessment. Two central characteristics of climate change are uncertainty and that it is a dynamic problem in which international actions can significantly alter climate projections and information needs, including partial and full compliance of global climate goals. Here we present AIRCC-Clim, a simple climate model emulator that produces regional probabilistic climate change projections of monthly and annual temperature and precipitation, as well as risk measures, based both on standard and user-defined emissions scenarios for six greenhouse gases. AIRCC-Clim emulates 37 atmosphere-ocean coupled general circulation models with low computational and technical requirements for the user. This standalone, user-friendly software is designed for a variety of applications including impact assessments, climate policy evaluation and integrated assessment modelling.</summary>\\n    <category term=\\\"physics.ao-ph\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"econ.GN\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"stat.AP\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <published>2021-10-30T11:09:03Z</published>\\n    <arxiv:primary_category term=\\\"physics.ao-ph\\\"/>\\n    <author>\\n      <name>Francisco Estrada</name>\\n    </author>\\n    <author>\\n      <name>Oscar Calder\\u00f3n-Bustamante</name>\\n    </author>\\n    <author>\\n      <name>Wouter Botzen</name>\\n    </author>\\n    <author>\\n      <name>Juli\\u00e1n A. Velasco</name>\\n    </author>\\n    <author>\\n      <name>Richard S. J. Tol</name>\\n    </author>\\n  </entry>\\n</feed>\\n\", \"https://export.arxiv.org/api/query?max_results=3&search_query=all%3Aprotein+folding&sortBy=relevance&start=0\": \"<?xml version='1.0' encoding='UTF-8'?>\\n<feed xmlns:opensearch=\\\"http://a9.com/-/spec/opensearch/1.1/\\\" xmlns:arxiv=\\\"http://arxiv.org/schemas/atom\\\" xmlns=\\\"http://www.w3.org/2005/Atom\\\">\\n  <id>https://arxiv.org/api/RyN4RSjCPjjHfjZ0UcYNZo4VQB8</id>\\n  <title>arXiv Query: search_query=all:protein OR all:folding&amp;id_list=&amp;start=0&amp;max_results=3</title>\\n  <updated>2026-11-21T18:42:15Z</updated>\\n  <link href=\\\"https://arxiv.org/api/query?search_query=all:protein+OR+all:folding&amp;start=0&amp;max_results=3&amp;id_list=\\\" type=\\\"application/atom+xml\\\"/>\\n  <opensearch:itemsPerPage>3</opensearch:itemsPerPage>\\n  <opensearch:totalResults>40493</opensearch:totalResults>\\n  <opensearch:startIndex>0</opensearch:startIndex>\\n  <entry>\\n    <id>http://arxiv.org/abs/cond-mat/9812291v1</id>\\n    <title>Discrete molecular dynamics studies of the folding of a protein-like model</title>\\n    <updated>1998-12-17T02:15:13Z</updated>\\n    <link href=\\\"https://arxiv.org/abs/cond-mat/9812291v1\\\" rel=\\\"alternate\\\" type=\\\"text/html\\\"/>\\n    <link href=\\\"https://arxiv.org/pdf/cond-mat/9812291v1\\\" rel=\\\"related\\\" type=\\\"application/pdf\\\" title=\\\"pdf\\\"/>\\n    <summary>  Background: Many attempts have been made to resolve in time the folding of model proteins in computer simulations. Different computational approaches have emerged. Some of these approaches suffer from the insensitivity to the geometrical properties of the proteins (lattice models), while others are computationally heavy (traditional MD).\\n  Results: We use a recently-proposed approach of Zhou and Karplus to study the folding of the protein model based on the discrete time molecular dynamics algorithm. We show that this algorithm resolves with respect to time the folding --- unfolding transition. In addition, we demonstrate the ability to study the coreof the model protein.\\n  Conclusion: The algorithm along with the model of inter-residue interactions can serve as a tool to study the thermodynamics and kinetics of protein models.</summary>\\n    <category term=\\\"cond-mat.stat-mech\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"q-bio\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <published>1998-12-17T02:15:13Z</published>\\n    <arxiv:comment>15 pages including 20 figures (Folding &amp; Design in press)</arxiv:comment>\\n    <arxiv:primary_category term=\\\"cond-mat.stat-mech\\\"/>\\n    <arxiv:journal_ref>Folding &amp; Design 3: 577--587 (1998)</arxiv:journal_ref>\\n    <author>\\n      <name>Nikolay V. Dokholyan</name>\\n      <arxiv:affiliation>Center for Polymer Studies, Physics Department, Boston University, Boston</arxiv:affiliation>\\n    </author>\\n    <author>\\n      <name>Sergey V. Buldyrev</name>\\n      <arxiv:affiliation>Center for Polymer Studies, Physics Department, Boston University, Boston</arxiv:affiliation>\\n    </author>\\n    <author>\\n      <name>H. Eugene Stanley</name>\\n      <arxiv:affiliation>Center for Polymer Studies, Physics Department, Boston University, Boston</arxiv:affiliation>\\n    </author>\\n    <author>\\n      <name>Eugene I. Shakhnovich</name>\\n      <arxiv:affiliation>Department of Chemistry, Harvard University, Cambridge</arxiv:affiliation>\\n    </author>\\n  </entry>\\n  <entry>\\n    <id>http://arxiv.org/abs/1002.5013v1</id>\\n    <title>Protein Folding: A Perspective From Statistical Physics</title>\\n    <updated>2010-02-26T15:49:03Z</updated>\\n    <link href=\\\"https://arxiv.org/abs/1002.5013v1\\\" rel=\\\"alternate\\\" type=\\\"text/html\\\"/>\\n    <link href=\\\"https://arxiv.org/pdf/1002.5013v1\\\" rel=\\\"related\\\" type=\\\"application/pdf\\\" title=\\\"pdf\\\"/>\\n    <summary>  In this paper, we introduce an approach to the protein folding problem from the point of view of statistical physics. Protein folding is a stochastic process by which a polypeptide folds into its characteristic and functional 3D structure from random coil. The process involves an intricate interplay between global geometry and local structure, and each protein seems to present special problems. We introduce CSAW (conditioned self-avoiding walk), a model of protein folding that combines the features of self-avoiding walk (SAW) and the Monte Carlo method. In this model, the unfolded protein chain is treated as a random coil described by SAW. Folding is induced by hydrophobic forces and other interactions, such as hydrogen bonding, which can be taken into account by imposing conditions on SAW. Conceptually, the mathematical basis is a generalized Langevin equation. To illustrate the flexibility and capabilities of the model, we consider several examples, including helix formation, elastic properties, and the transition in the folding of myoglobin. From the CSAW simulation and physical arguments, we find a universal elastic energy for proteins, which depends only on the radius of gyration $R_{g}$ and the residue number $N$. The elastic energy gives rise to scaling laws $R_{g}\\\\sim N^\\u03bd$ in different regions with exponents $\\u03bd=3/5,3/7,2/5$, consistent with the observed unfolded stage, pre-globule, and molten globule, respectively. These results indicate that CSAW can serve as a theoretical laboratory to study universal principles in protein folding.</summary>\\n    <category term=\\\"cond-mat.stat-mech\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"cond-mat.soft\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"physics.bio-ph\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"q-bio.BM\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <published>2010-02-26T15:49:03Z</published>\\n    <arxiv:primary_category term=\\\"cond-mat.stat-mech\\\"/>\\n    <arxiv:journal_ref>Protein Folding, Eric C. Walters ed. NOVA Publishers, 2010, pp. 579-604</arxiv:journal_ref>\\n    <author>\\n      <name>Jinzhi Lei</name>\\n    </author>\\n    <author>\\n      <name>Kerson Huang</name>\\n    </author>\\n  </entry>\\n  <entry>\\n    <id>http://arxiv.org/abs/cond-mat/9806323v1</id>\\n    <title>Testing a New Monte Carlo Strategy for Folding Model Proteins</title>\\n    <updated>1998-06-26T10:03:19Z</updated>\\n    <link href=\\\"https://arxiv.org/abs/cond-mat/9806323v1\\\" rel=\\\"alternate\\\" type=\\\"text/html\\\"/>\\n    <link href=\\\"https://arxiv.org/pdf/cond-mat/9806323v1\\\" rel=\\\"related\\\" type=\\\"application/pdf\\\" title=\\\"pdf\\\"/>\\n    <summary>  We demonstrate that the recently proposed pruned-enriched Rosenbluth method PERM (P.~Grassberger, Phys.~Rev.~{\\\\bf E 56} (1997)\\n  3682) leads to very efficient algorithms for the folding of simple model proteins. We test it on several models for lattice heteropolymers, and compare to published Monte Carlo studies of the properties of particular sequences. In all cases our method is faster than the previous ones, and in several cases we find new minimal energy states. In addition to producing more reliable candidates for ground states, our method gives detailed information about the thermal spectrum and, thus, allows to analyze static aspects of the folding behavior of arbitrary sequences.</summary>\\n    <category term=\\\"cond-mat.soft\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"q-bio\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <published>1998-06-26T10:03:19Z</published>\\n    <arxiv:comment>17 pages, 21 figures, needs mprocl.sty (included), appears in Workshop on: Monte Carlo Approach to Biopolymers and Protein Folding, World Scientific 1998</arxiv:comment>\\n    <arxiv:primary_category term=\\\"cond-mat.soft\\\"/>\\n    <author>\\n      <name>H. Frauenkron</name>\\n      <arxiv:affiliation>HLRZ c/o Forschungszentrum J\\u00fclich, Germany;</arxiv:affiliation>\\n    </author>\\n    <author>\\n      <name>U. Bastolla</name>\\n      <arxiv:affiliation>HLRZ c/o Forschungszentrum J\\u00fclich, Germany;</arxiv:affiliation>\\n    </author>\\n    <author>\\n      <name>E. Gerstner</name>\\n      <arxiv:affiliation>HLRZ c/o Forschungszentrum J\\u00fclich, Germany;</arxiv:affiliation>\\n      <arxiv:affiliation>Physics Department, University of Wuppertal,Germany</arxiv:affiliation>\\n    </author>\\n    <author>\\n      <name>P. Grassberger</name>\\n      <arxiv:affiliation>HLRZ c/o Forschungszentrum J\\u00fclich, Germany;</arxiv:affiliation>\\n      <arxiv:affiliation>Physics Department, University of Wuppertal,Germany</arxiv:affiliation>\\n    </author>\\n    <author>\\n      <name>und W. Nadler</name>\\n      <arxiv:affiliation>HLRZ c/o Forschungszentrum J\\u00fclich, Germany;</arxiv:affiliation>\\n    </author>\\n  </entry>\\n</feed>\\n\", \"https://export.arxiv.org/api/query?max_results=3&search_query=all%3Aprotein+structure+prediction+deep+learning&sortBy=relevance&start=0\": \"<?xml version='1.0' encoding='UTF-8'?>\\n<feed xmlns:opensearch=\\\"http://a9.com/-/spec/opensearch/1.1/\\\" xmlns:arxiv=\\\"http://arxiv.org/schemas/atom\\\" xmlns=\\\"http://www.w3.org/2005/Atom\\\">\\n  <id>https://arxiv.org/api/wr1iZQMF5OO5VQeTTcQi2CttxCE</id>\\n  <title>arXiv Query: search_query=all:protein OR all:structure OR all:prediction OR all:deep OR all:learning&amp;id_list=&amp;start=0&amp;max_results=3</title>\\n  <updated>2026-11-21T18:40:28Z</updated>\\n  <link href=\\\"https://arxiv.org/api/query?search_query=all:protein+OR+(all:structure+OR+(all:prediction+OR+(all:deep+OR+all:learning)))&amp;start=0&amp;max_results=3&amp;id_list=\\\" type=\\\"application/atom+xml\\\"/>\\n  <opensearch:itemsPerPage>3</opensearch:itemsPerPage>\\n  <opensearch:totalResults>1203675</opensearch:totalResults>\\n  <opensearch:startIndex>0</opensearch:startIndex>\\n  <entry>\\n    <id>http://arxiv.org/abs/2306.11113v2</id>\\n    <title>Learn to Accumulate Evidence from All Training Samples: Theory and Practice</title>\\n    <updated>2023-06-24T17:45:51Z</updated>\\n    <link href=\\\"https://arxiv.org/abs/2306.11113v2\\\" rel=\\\"alternate\\\" type=\\\"text/html\\\"/>\\n    <link href=\\\"https://arxiv.org/pdf/2306.11113v2\\\" rel=\\\"related\\\" type=\\\"application/pdf\\\" title=\\\"pdf\\\"/>\\n    <summary>Evidential deep learning, built upon belief theory and subjective logic, offers a principled and computationally efficient way to turn a deterministic neural network uncertainty-aware. The resultant evidential models can quantify fine-grained uncertainty using the learned evidence. To ensure theoretically sound evidential models, the evidence needs to be non-negative, which requires special activation functions for model training and inference. This constraint often leads to inferior predictive performance compared to standard softmax models, making it challenging to extend them to many large-scale datasets. To unveil the real cause of this undesired behavior, we theoretically investigate evidential models and identify a fundamental limitation that explains the inferior performance: existing evidential activation functions create zero evidence regions, which prevent the model to learn from training samples falling into such regions. A deeper analysis of evidential activation functions based on our theoretical underpinning inspires the design of a novel regularizer that effectively alleviates this fundamental limitation. Extensive experiments over many challenging real-world datasets and settings confirm our theoretical findings and demonstrate the effectiveness of our proposed approach.</summary>\\n    <category term=\\\"cs.LG\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"cs.AI\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"cs.CV\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <published>2023-06-19T18:27:12Z</published>\\n    <arxiv:comment>ICML 2023; Analysis and improvement of evidential deep learning theory</arxiv:comment>\\n    <arxiv:primary_category term=\\\"cs.LG\\\"/>\\n    <author>\\n      <name>Deep Pandey</name>\\n    </author>\\n    <author>\\n      <name>Qi Yu</name>\\n    </author>\\n  </entry>\\n  <entry>\\n    <id>http://arxiv.org/abs/2105.04026v2</id>\\n    <title>The Modern Mathematics of Deep Learning</title>\\n    <updated>2023-02-08T16:48:49Z</updated>\\n    <link href=\\\"https://arxiv.org/abs/2105.04026v2\\\" rel=\\\"alternate\\\" type=\\\"text/html\\\"/>\\n    <link href=\\\"https://arxiv.org/pdf/2105.04026v2\\\" rel=\\\"related\\\" type=\\\"application/pdf\\\" title=\\\"pdf\\\"/>\\n    <summary>We describe the new field of mathematical analysis of deep learning. This field emerged around a list of research questions that were not answered within the classical framework of learning theory. These questions concern: the outstanding generalization power of overparametrized neural networks, the role of depth in deep architectures, the apparent absence of the curse of dimensionality, the surprisingly successful optimization performance despite the non-convexity of the problem, understanding what features are learned, why deep architectures perform exceptionally well in physical problems, and which fine aspects of an architecture affect the behavior of a learning task in which way. We present an overview of modern approaches that yield partial answers to these questions. For selected approaches, we describe the main ideas in more detail.</summary>\\n    <category term=\\\"cs.LG\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"stat.ML\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <published>2021-05-09T21:30:42Z</published>\\n    <arxiv:comment>A version of this review paper appears as a chapter in the book \\\"Mathematical Aspects of Deep Learning\\\" by Cambridge University Press</arxiv:comment>\\n    <arxiv:primary_category term=\\\"cs.LG\\\"/>\\n    <arxiv:journal_ref>Mathematical Aspects of Deep Learning, pp. 1-111. Cambridge University Press, 2022</arxiv:journal_ref>\\n    <author>\\n      <name>Julius Berner</name>\\n    </author>\\n    <author>\\n      <name>Philipp Grohs</name>\\n    </author>\\n    <author>\\n      <name>Gitta Kutyniok</name>\\n    </author>\\n    <author>\\n      <name>Philipp Petersen</name>\\n    </author>\\n    <arxiv:doi>10.1017/9781009025096.002</arxiv:doi>\\n    <link rel=\\\"related\\\" href=\\\"https://doi.org/10.1017/9781009025096.002\\\" title=\\\"doi\\\"/>\\n  </entry>\\n  <entry>\\n    <id>http://arxiv.org/abs/1605.03004v1</id>\\n    <title>MUST-CNN: A Multilayer Shift-and-Stitch Deep Convolutional Architecture for Sequence-based Protein Structure Prediction</title>\\n    <updated>2016-05-10T13:31:52Z</updated>\\n    <link href=\\\"https://arxiv.org/abs/1605.03004v1\\\" rel=\\\"alternate\\\" type=\\\"text/html\\\"/>\\n    <link href=\\\"https://arxiv.org/pdf/1605.03004v1\\\" rel=\\\"related\\\" type=\\\"application/pdf\\\" title=\\\"pdf\\\"/>\\n    <summary>Predicting protein properties such as solvent accessibility and secondary structure from its primary amino acid sequence is an important task in bioinformatics. Recently, a few deep learning models have surpassed the traditional window based multilayer perceptron. Taking inspiration from the image classification domain we propose a deep convolutional neural network architecture, MUST-CNN, to predict protein properties. This architecture uses a novel multilayer shift-and-stitch (MUST) technique to generate fully dense per-position predictions on protein sequences. Our model is significantly simpler than the state-of-the-art, yet achieves better results. By combining MUST and the efficient convolution operation, we can consider far more parameters while retaining very fast prediction speeds. We beat the state-of-the-art performance on two large protein property prediction datasets.</summary>\\n    <category term=\\\"cs.LG\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <published>2016-05-10T13:31:52Z</published>\\n    <arxiv:comment>8 pages ; 3 figures ; deep learning based sequence-sequence prediction. in AAAI 2016</arxiv:comment>\\n    <arxiv:primary_category term=\\\"cs.LG\\\"/>\\n    <author>\\n      <name>Zeming Lin</name>\\n    </author>\\n    <author>\\n      <name>Jack Lanchantin</name>\\n    </author>\\n    <author>\\n      <name>Yanjun Qi</name>\\n    </author>\\n  </entry>\\n</feed>\\n\", \"https://export.arxiv.org/api/query?max_results=3&search_query=all%3Aquantum+error+correction&sortBy=relevance&start=0\": \"<?xml version='1.0' encoding='UTF-8'?>\\n<feed xmlns:opensearch=\\\"http://a9.com/-/spec/opensearch/1.1/\\\" xmlns:arxiv=\\\"http://arxiv.org/schemas/atom\\\" xmlns=\\\"http://www.w3.org/2005/Atom\\\">\\n  <id>https://arxiv.org/api/ZCLvMLTrVZEp1f7sFes9stV9ecQ</id>\\n  <title>arXiv Query: search_query=all:quantum OR all:error OR all:correction&amp;id_list=&amp;start=0&amp;max_results=3</title>\\n  <updated>2026-11-21T18:42:25Z</updated>\\n  <link href=\\\"https://arxiv.org/api/query?search_query=all:quantum+OR+(all:error+OR+all:correction)&amp;start=0&amp;max_results=3&amp;id_list=\\\" type=\\\"application/atom+xml\\\"/>\\n  <opensearch:itemsPerPage>3</opensearch:itemsPerPage>\\n  <opensearch:totalResults>759241</opensearch:totalResults>\\n  <opensearch:startIndex>0</opensearch:startIndex>\\n  <entry>\\n    <id>http://arxiv.org/abs/1311.2485v2</id>\\n    <title>Continuous-time quantum error correction</title>\\n    <updated>2013-12-01T00:04:10Z</updated>\\n    <link href=\\\"https://arxiv.org/abs/1311.2485v2\\\" rel=\\\"alternate\\\" type=\\\"text/html\\\"/>\\n    <link href=\\\"https://arxiv.org/pdf/1311.2485v2\\\" rel=\\\"related\\\" type=\\\"application/pdf\\\" title=\\\"pdf\\\"/>\\n    <summary>Continuous-time quantum error correction (CTQEC) is an approach to protecting quantum information from noise in which both the noise and the error correcting operations are treated as processes that are continuous in time. This chapter investigates CTQEC based on continuous weak measurements and feedback from the point of view of the subsystem principle, which states that protected quantum information is contained in a subsystem of the Hilbert space. We study how to approach the problem of constructing CTQEC protocols by looking at the evolution of the state of the system in an encoded basis in which the subsystem containing the protected information is explicit. This point of view allows us to reduce the problem to that of protecting a known state, and to design CTQEC procedures from protocols for the protection of a single qubit. We show how previously studied CTQEC schemes with both direct and indirect feedback can be obtained from strategies for the protection of a single qubit via weak measurements and weak unitary operations. We also review results on the performance of CTQEC with direct feedback in cases of Markovian and non-Markovian decoherence, where we have shown that due to the existence of a Zeno regime in non-Markovian dynamics, the performance of CTQEC can exhibit a quadratic improvement if the time resolution of the weak error-correcting operations is high enough to reveal the non-Markovian character of the noise process.</summary>\\n    <category term=\\\"quant-ph\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <published>2013-11-11T16:34:10Z</published>\\n    <arxiv:comment>22 pages, 3 figures, this is a chapter in the book \\\"Quantum Error Correction\\\", edited by Daniel A. Lidar and Todd A. Brun, (Cambridge University Press, 2013), at http://www.cambridge.org/us/academic/subjects/physics/quantum-physics-quantum-information-and-quantum-computation/quantum-error-correction</arxiv:comment>\\n    <arxiv:primary_category term=\\\"quant-ph\\\"/>\\n    <arxiv:journal_ref>In \\\"Quantum Error Correction\\\", edited by Daniel A. Lidar and Todd A. Brun, (Cambridge University Press, 2013)</arxiv:journal_ref>\\n    <author>\\n      <name>Ognyan Oreshkov</name>\\n    </author>\\n  </entry>\\n  <entry>\\n    <id>http://arxiv.org/abs/1610.04013v1</id>\\n    <title>Entanglement-Assisted Quantum Error-Correcting Codes</title>\\n    <updated>2016-10-13T10:31:34Z</updated>\\n    <link href=\\\"https://arxiv.org/abs/1610.04013v1\\\" rel=\\\"alternate\\\" type=\\\"text/html\\\"/>\\n    <link href=\\\"https://arxiv.org/pdf/1610.04013v1\\\" rel=\\\"related\\\" type=\\\"application/pdf\\\" title=\\\"pdf\\\"/>\\n    <summary>We provide a self-contained introduction for entanglement-assisted quantum error-correcting codes in this book chapter.</summary>\\n    <category term=\\\"quant-ph\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <published>2016-10-13T10:31:34Z</published>\\n    <arxiv:comment>This is a chapter in the book \\\"Quantum Error Correction\\\", edited by Daniel A. Lidar and Todd A. Brun, (Cambridge University Press, 2013)</arxiv:comment>\\n    <arxiv:primary_category term=\\\"quant-ph\\\"/>\\n    <arxiv:journal_ref>In \\\"Quantum Error Correction\\\", edited by Daniel A. Lidar and Todd A. Brun, Cambridge University Press, 2013 (ISBN: 9780521897877)</arxiv:journal_ref>\\n    <author>\\n      <name>Todd Brun</name>\\n    </author>\\n    <author>\\n      <name>Min-Hsiu Hsieh</name>\\n    </author>\\n  </entry>\\n  <entry>\\n    <id>http://arxiv.org/abs/quant-ph/0602157v1</id>\\n    <title>An Introduction to Error-Correcting Codes: From Classical to Quantum</title>\\n    <updated>2006-02-18T03:17:50Z</updated>\\n    <link href=\\\"https://arxiv.org/abs/quant-ph/0602157v1\\\" rel=\\\"alternate\\\" type=\\\"text/html\\\"/>\\n    <link href=\\\"https://arxiv.org/pdf/quant-ph/0602157v1\\\" rel=\\\"related\\\" type=\\\"application/pdf\\\" title=\\\"pdf\\\"/>\\n    <summary>  This report surveys quantum error-correcting codes. As Preskill claimed, 21st century would be the golden age of quantum error correction. Quantum channels behave differently from classical channels, so researchers face difficulties in developing robust quantum codes. Fortunately, the classical error control methods have been well developed. If we can learn many lessons from classical coding theory, we can expedite the development of quantum codes. Scientists have discovered that quantum error correction shares many concepts with classical counterpart. Both quantum and classical coding schemes add redundancy to information to protect against noises. They also have similar conditions for error detectability and correctability.</summary>\\n    <category term=\\\"quant-ph\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <published>2006-02-18T03:17:50Z</published>\\n    <arxiv:comment>This is a brief introduction to quantum error correction schemes. Comments are welcome</arxiv:comment>\\n    <arxiv:primary_category term=\\\"quant-ph\\\"/>\\n    <author>\\n      <name>Hsun-Hsien Chang</name>\\n    </author>\\n  </entry>\\n</feed>\\n\", \"https://export.arxiv.org/api/query?max_results=3&search_query=all%3Atransformer+attention+mechanism&sortBy=relevance&start=0\": \"<?xml version='1.0' encoding='UTF-8'?>\\n<feed xmlns:opensearch=\\\"http://a9.com/-/spec/opensearch/1.1/\\\" xmlns:arxiv=\\\"http://arxiv.org/schemas/atom\\\" xmlns=\\\"http://www.w3.org/2005/Atom\\\">\\n  <id>https://arxiv.org/api/Pt9teuoGGrHQ1CnfckS4BtEmZho</id>\\n  <title>arXiv Query: search_query=all:transformer OR all:attention OR all:mechanism&amp;id_list=&amp;start=0&amp;max_results=3</title>\\n  <updated>2026-11-21T18:42:22Z</updated>\\n  <link href=\\\"https://arxiv.org/api/query?search_query=all:transformer+OR+(all:attention+OR+all:mechanism)&amp;start=0&amp;max_results=3&amp;id_list=\\\" type=\\\"application/atom+xml\\\"/>\\n  <opensearch:itemsPerPage>3</opensearch:itemsPerPage>\\n  <opensearch:totalResults>505933</opensearch:totalResults>\\n  <opensearch:startIndex>0</opensearch:startIndex>\\n  <entry>\\n    <id>http://arxiv.org/abs/2206.03003v2</id>\\n    <title>Transformer-based Personalized Attention Mechanism for Medical Images with Clinical Records</title>\\n    <updated>2023-01-27T09:54:07Z</updated>\\n    <link href=\\\"https://arxiv.org/abs/2206.03003v2\\\" rel=\\\"alternate\\\" type=\\\"text/html\\\"/>\\n    <link href=\\\"https://arxiv.org/pdf/2206.03003v2\\\" rel=\\\"related\\\" type=\\\"application/pdf\\\" title=\\\"pdf\\\"/>\\n    <summary>In medical image diagnosis, identifying the attention region, i.e., the region of interest for which the diagnosis is made, is an important task. Various methods have been developed to automatically identify target regions from given medical images. However, in actual medical practice, the diagnosis is made based not only on the images but also on a variety of clinical records. This means that pathologists examine medical images with some prior knowledge of the patients and that the attention regions may change depending on the clinical records. In this study, we propose a method called the Personalized Attention Mechanism (PersAM), by which the attention regions in medical images are adaptively changed according to the clinical records. The primary idea of the PersAM method is to encode the relationships between the medical images and clinical records using a variant of Transformer architecture. To demonstrate the effectiveness of the PersAM method, we applied it to a large-scale digital pathology problem of identifying the subtypes of 842 malignant lymphoma patients based on their gigapixel whole slide images and clinical records.</summary>\\n    <category term=\\\"eess.IV\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"cs.CV\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <published>2022-06-07T04:35:22Z</published>\\n    <arxiv:primary_category term=\\\"eess.IV\\\"/>\\n    <arxiv:journal_ref>Takagi, Yusuke, et al. \\\"Transformer-based personalized attention mechanism for medical images with clinical records.\\\" Journal of Pathology Informatics (2023): 100185</arxiv:journal_ref>\\n    <author>\\n      <name>Yusuke Takagi</name>\\n    </author>\\n    <author>\\n      <name>Noriaki Hashimoto</name>\\n    </author>\\n    <author>\\n      <name>Hiroki Masuda</name>\\n    </author>\\n    <author>\\n      <name>Hiroaki Miyoshi</name>\\n    </author>\\n    <author>\\n      <name>Koichi Ohshima</name>\\n    </author>\\n    <author>\\n      <name>Hidekata Hontani</name>\\n    </author>\\n    <author>\\n      <name>Ichiro Takeuchi</name>\\n    </author>\\n  </entry>\\n  <entry>\\n    <id>http://arxiv.org/abs/2209.15001v3</id>\\n    <title>Dilated Neighborhood Attention Transformer</title>\\n    <updated>2023-01-16T18:58:58Z</updated>\\n    <link href=\\\"https://arxiv.org/abs/2209.15001v3\\\" rel=\\\"alternate\\\" type=\\\"text/html\\\"/>\\n    <link href=\\\"https://arxiv.org/pdf/2209.15001v3\\\" rel=\\\"related\\\" type=\\\"application/pdf\\\" title=\\\"pdf\\\"/>\\n    <summary>Transformers are quickly becoming one of the most heavily applied deep learning architectures across modalities, domains, and tasks. In vision, on top of ongoing efforts into plain transformers, hierarchical transformers have also gained significant attention, thanks to their performance and easy integration into existing frameworks. These models typically employ localized attention mechanisms, such as the sliding-window Neighborhood Attention (NA) or Swin Transformer's Shifted Window Self Attention. While effective at reducing self attention's quadratic complexity, local attention weakens two of the most desirable properties of self attention: long range inter-dependency modeling, and global receptive field. In this paper, we introduce Dilated Neighborhood Attention (DiNA), a natural, flexible and efficient extension to NA that can capture more global context and expand receptive fields exponentially at no additional cost. NA's local attention and DiNA's sparse global attention complement each other, and therefore we introduce Dilated Neighborhood Attention Transformer (DiNAT), a new hierarchical vision transformer built upon both. DiNAT variants enjoy significant improvements over strong baselines such as NAT, Swin, and ConvNeXt. Our large model is faster and ahead of its Swin counterpart by 1.6% box AP in COCO object detection, 1.4% mask AP in COCO instance segmentation, and 1.4% mIoU in ADE20K semantic segmentation. Paired with new frameworks, our large variant is the new state of the art panoptic segmentation model on COCO (58.5 PQ) and ADE20K (49.4 PQ), and instance segmentation model on Cityscapes (45.1 AP) and ADE20K (35.4 AP) (no extra data). It also matches the state of the art specialized semantic segmentation models on ADE20K (58.1 mIoU), and ranks second on Cityscapes (84.5 mIoU) (no extra data).</summary>\\n    <category term=\\\"cs.CV\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"cs.AI\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"cs.LG\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <published>2022-09-29T17:57:08Z</published>\\n    <arxiv:comment>Large results were updated according to the new checkpoint. We open-source our project at https://github.com/SHI-Labs/Neighborhood-Attention-Transformer</arxiv:comment>\\n    <arxiv:primary_category term=\\\"cs.CV\\\"/>\\n    <author>\\n      <name>Ali Hassani</name>\\n    </author>\\n    <author>\\n      <name>Humphrey Shi</name>\\n    </author>\\n  </entry>\\n  <entry>\\n    <id>http://arxiv.org/abs/1809.04281v3</id>\\n    <title>Music Transformer</title>\\n    <updated>2018-12-12T07:42:08Z</updated>\\n    <link href=\\\"https://arxiv.org/abs/1809.04281v3\\\" rel=\\\"alternate\\\" type=\\\"text/html\\\"/>\\n    <link href=\\\"https://arxiv.org/pdf/1809.04281v3\\\" rel=\\\"related\\\" type=\\\"application/pdf\\\" title=\\\"pdf\\\"/>\\n    <summary>Music relies heavily on repetition to build structure and meaning. Self-reference occurs on multiple timescales, from motifs to phrases to reusing of entire sections of music, such as in pieces with ABA structure. The Transformer (Vaswani et al., 2017), a sequence model based on self-attention, has achieved compelling results in many generation tasks that require maintaining long-range coherence. This suggests that self-attention might also be well-suited to modeling music. In musical composition and performance, however, relative timing is critically important. Existing approaches for representing relative positional information in the Transformer modulate attention based on pairwise distance (Shaw et al., 2018). This is impractical for long sequences such as musical compositions since their memory complexity for intermediate relative information is quadratic in the sequence length. We propose an algorithm that reduces their intermediate memory requirement to linear in the sequence length. This enables us to demonstrate that a Transformer with our modified relative attention mechanism can generate minute-long compositions (thousands of steps, four times the length modeled in Oore et al., 2018) with compelling structure, generate continuations that coherently elaborate on a given motif, and in a seq2seq setup generate accompaniments conditioned on melodies. We evaluate the Transformer with our relative attention mechanism on two datasets, JSB Chorales and Piano-e-Competition, and obtain state-of-the-art results on the latter.</summary>\\n    <category term=\\\"cs.LG\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"cs.SD\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"eess.AS\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <category term=\\\"stat.ML\\\" scheme=\\\"http://arxiv.org/schemas/atom\\\"/>\\n    <published>2018-09-12T07:15:26Z</published>\\n    <arxiv:comment>Improved skewing section and accompanying figures. Previous titles are \\\"An Improved Relative Self-Attention Mechanism for Transformer with Application to Music Generation\\\" and \\\"Music Transformer\\\"</arxiv:comment>\\n    <arxiv:primary_category term=\\\"cs.LG\\\"/>\\n    <author>\\n      <name>Cheng-Zhi Anna Huang</name>\\n    </author>\\n    <author>\\n      <name>Ashish Vaswani</name>\\n    </author>\\n    <author>\\n      <name>Jakob Uszkoreit</name>\\n    </author>\\n    <author>\\n      <name>Noam Shazeer</name>\\n    </author>\\n    <author>\\n      <name>Ian Simon</name>\\n    </author>\\n    <author>\\n      <name>Curtis Hawthorne</name>\\n    </author>\\n    <author>\\n      <name>Andrew M. Dai</name>\\n    </author>\\n    <author>\\n      <name>Matthew D. Hoffman</name>\\n    </author>\\n    <author>\\n      <name>Monica Dinculescu</name>\\n    </author>\\n    <author>\\n      <name>Douglas Eck</name>\\n    </author>\\n  </entry>\\n</feed>\\n\", \"https://export.arxiv.org/api/query?max_results=3&search_query=all%3Azzzzqqqxxkk&sortBy=relevance&start=0\": \"<?xml version='1.0' encoding='UTF-8'?>\\n<feed xmlns:opensearch=\\\"http://a9.com/-/spec/opensearch/1.1/\\\" xmlns:arxiv=\\\"http://arxiv.org/schemas/atom\\\" xmlns=\\\"http://www.w3.org/2005/Atom\\\">\\n  <id>https://arxiv.org/api/TOGxplvAmNKamk2rLN9KxadLHss</id>\\n  <title>arXiv Query: search_query=all:zzzzqqqxxkk&amp;id_list=&amp;start=0&amp;max_results=3</title>\\n  <updated>2026-11-21T18:42:27Z</updated>\\n  <link href=\\\"https://arxiv.org/api/query?search_query=all:zzzzqqqxxkk&amp;start=0&amp;max_results=3&amp;id_list=\\\" type=\\\"application/atom+xml\\\"/>\\n  <opensearch:itemsPerPage>3</opensearch:itemsPerPage>\\n  <opensearch:totalResults>0</opensearch:totalResults>\\n  <opensearch:startIndex>0</opensearch:startIndex>\\n</feed>\\n\"}"))

In [ ]:
OFFLINE = False

client = make_client()
assert smoke_test(client), "Fix the key problem above, then run this cell again."

## 4. Tell the model what the tools do

The model sees only these descriptions, not our Python code, so it picks
tools based on them.

In [ ]:
"""Tool schemas sent to the model, with editable descriptions."""

SEARCH_ARXIV_DESCRIPTION = (
    "Search arXiv for research papers on a topic. Returns the id, title, "
    "authors, date, and link of the best matches."
)
GET_ABSTRACT_DESCRIPTION = (
    "Get the abstract of one arXiv paper. Use an id returned by search_arxiv."
)
LOOKUP_WIKIPEDIA_DESCRIPTION = (
    "Look up a term or concept on Wikipedia and get a short plain-language "
    "summary. Use it to explain jargon."
)

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_arxiv",
            "description": SEARCH_ARXIV_DESCRIPTION,
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search terms"},
                    "max_results": {
                        "type": "integer",
                        "description": "How many papers to return, 1 to 3",
                    },
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_abstract",
            "description": GET_ABSTRACT_DESCRIPTION,
            "parameters": {
                "type": "object",
                "properties": {
                    "paper_id": {
                        "type": "string",
                        "description": "An arXiv id such as 2101.00001",
                    },
                },
                "required": ["paper_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_wikipedia",
            "description": LOOKUP_WIKIPEDIA_DESCRIPTION,
            "parameters": {
                "type": "object",
                "properties": {
                    "term": {"type": "string", "description": "Word or phrase"},
                },
                "required": ["term"],
            },
        },
    },
]

## 5. The agent loop

The loop asks the model. If it wants a tool, we run the tool, hand back the
result, and ask again. When it answers without a tool request, we are done.
`MAX_STEPS` stops a run that never settles, and the last step turns tools
off so the model has to answer.

In [ ]:
# @title Trace printer and citation check
"""Print each agent step and collect tokens, latency, and raw tool results."""

import json

DISPLAY_CHARS = 300


class Tracer:
    """Record and print one agent run."""

    def __init__(self, verbose: bool = True) -> None:
        self.verbose = verbose
        self.calls: list[dict] = []
        self.tool_results: list[dict] = []
        self.final_answer: str | None = None

    def _print(self, text: str) -> None:
        if self.verbose:
            print(text)

    def model_call(
        self,
        step: int,
        tokens: int,
        seconds: float,
        prompt: int = 0,
        completion: int = 0,
        cached: int = 0,
        reasoning: int = 0,
        served_by: str = "",
        cost: float = 0.0,
    ) -> None:
        """Record the token counts and latency of one model call.

        ``cached`` is part of ``prompt`` and ``reasoning`` is part of
        ``completion``. Both stay 0 when the API does not report them.
        ``served_by`` is the upstream provider OpenRouter names in its reply and
        ``cost`` is the dollar cost it reports, when usage accounting is on.
        """
        self.calls.append(
            {
                "step": step,
                "tokens": tokens,
                "seconds": seconds,
                "prompt": prompt,
                "completion": completion,
                "cached": cached,
                "reasoning": reasoning,
                "served_by": served_by,
                "cost": cost,
            }
        )

    def tool_call(self, step: int, name: str, args: dict, result) -> None:
        """Print one tool call and keep the raw result for the citation check."""
        self.tool_results.append({"step": step, "name": name, "args": args, "result": result})
        shown = result if isinstance(result, str) else json.dumps(result)
        if len(shown) > DISPLAY_CHARS:
            shown = shown[:DISPLAY_CHARS] + "..."
        self._print(f"Step {step}: {name}({json.dumps(args)})")
        self._print(f"  -> {shown}")

    def note(self, text: str) -> None:
        """Print a line about something that went wrong mid-run."""
        self._print(text)

    def answer(self, text: str) -> None:
        """Print and keep the final answer."""
        self.final_answer = text
        self._print(f"\nAnswer:\n{text}")

    def summary(self) -> dict:
        """Return totals for the run."""
        return {
            "model_calls": len(self.calls),
            "tool_calls": len(self.tool_results),
            "tokens": sum(c["tokens"] for c in self.calls),
            "prompt_tokens": sum(c["prompt"] for c in self.calls),
            "completion_tokens": sum(c["completion"] for c in self.calls),
            "cached_tokens": sum(c["cached"] for c in self.calls),
            "reasoning_tokens": sum(c["reasoning"] for c in self.calls),
            "cost": round(sum(c["cost"] for c in self.calls), 6),
            "served_by": " ".join(sorted({c["served_by"] for c in self.calls} - {""})),
            "seconds": round(sum(c["seconds"] for c in self.calls), 2),
        }
"""Check that every paper an answer cites was returned by a tool in this run."""

import re
from dataclasses import dataclass, field


NEW_ID = r"(?<![\d.])\d{4}\.\d{4,5}(?:v\d+)?(?![\d])"
OLD_ID = r"(?<![\w-])[a-z]+(?:-[a-z]+)*(?:\.[A-Z]{2})?/\d{7}(?:v\d+)?"
ID_PATTERN = re.compile(f"{NEW_ID}|{OLD_ID}")
# Models often write the hyphen of an id like quant-ph/0602157 as a Unicode
# hyphen or en dash, which would cut the id short at the first letter group.
UNICODE_DASHES = "\u2010\u2011\u2012\u2013\u2212\ufe63\uff0d"
LOOSE_OLD_ID = re.compile(
    rf"(?<![\w{UNICODE_DASHES}-])[a-z]+(?:[-{UNICODE_DASHES}][a-z]+)*"
    r"(?:\.[A-Z]{2})?/\d{7}(?:v\d+)?"
)
DASH_CHARS = re.compile(f"[{UNICODE_DASHES}]")
QUOTED = re.compile(r"[\"“]([^\"”]+)[\"”]")
MIN_QUOTED_WORDS = 4
# Text with these characters is an example or a formula, not a paper title.
NOT_TITLE_CHARS = re.compile(r"[\u2190-\u21ff=<>+*|{}\[\]^_~\\]")


@dataclass
class CitationReport:
    verified: list[str] = field(default_factory=list)
    unverified: list[str] = field(default_factory=list)
    summary: str = ""


def _looks_like_title(phrase: str) -> bool:
    phrase = phrase.strip()
    return (
        bool(phrase)
        and phrase[0].isupper()
        and not phrase.endswith(".")
        and not NOT_TITLE_CHARS.search(phrase)
    )


def _norm_title(text: str) -> str:
    return " ".join(re.findall(r"[a-z0-9]+", text.lower()))


def _retrieved(tool_results: list[dict]) -> tuple[dict[str, str], set[str]]:
    # Wikipedia titles go in the title set only, so a quoted one is not flagged.
    papers: dict[str, str] = {}
    titles: set[str] = set()
    for entry in tool_results:
        result = entry["result"]
        items = result if isinstance(result, list) else [result]
        for item in items:
            if not isinstance(item, dict) or "title" not in item:
                continue
            title = _norm_title(item["title"])
            titles.add(title)
            if entry["name"] != "lookup_wikipedia" and "id" in item:
                papers[_normalize_id(item["id"])] = title
    return papers, titles


def check_citations(answer: str, tool_results: list[dict]) -> CitationReport:
    """Count a citation as verified only if a tool returned it in this run.

    Match papers by arXiv id, or by title when the answer gives no id. A
    quoted phrase of four or more words counts as a reference to any
    retrieved paper whose title contains it. It is unverified when no
    retrieved title does, unless it reads as an example rather than a title:
    it starts in lowercase, ends in a period, or has arrows or math symbols.
    """
    answer = LOOSE_OLD_ID.sub(lambda m: DASH_CHARS.sub("-", m.group()), answer)
    papers, titles = _retrieved(tool_results)
    report = CitationReport()
    seen_ids: set[str] = set()

    for match in ID_PATTERN.findall(answer):
        pid = _normalize_id(match)
        if pid in seen_ids:
            continue
        seen_ids.add(pid)
        (report.verified if pid in papers else report.unverified).append(pid)

    padded = f" {_norm_title(answer)} "
    for pid, title in papers.items():
        if pid not in seen_ids and f" {title} " in padded:
            seen_ids.add(pid)
            report.verified.append(pid)

    for quoted in QUOTED.findall(answer):
        title = _norm_title(quoted)
        if len(title.split()) < MIN_QUOTED_WORDS:
            continue
        needle = f" {title} "
        matched = [pid for pid, known in papers.items() if needle in f" {known} "]
        if matched:
            for pid in matched:
                if pid not in seen_ids:
                    seen_ids.add(pid)
                    report.verified.append(pid)
        elif _looks_like_title(quoted) and not any(needle in f" {known} " for known in titles):
            report.unverified.append(quoted.strip())

    total = len(report.verified) + len(report.unverified)
    if total == 0:
        report.summary = "The answer cites no papers."
    else:
        report.summary = (
            f"{len(report.verified)} of {total} cited papers "
            "were retrieved by the agent in this run."
        )
    return report

In [ ]:
"""The agent loop: ask the model, run the tools it picks, repeat."""

import inspect
import json
import time
from dataclasses import dataclass


MAX_STEPS = 6
RESULT_CHARS = 2000
TRUNCATION_MARK = " [truncated]"
EMPTY_ANSWER = "The model returned an empty answer."
EMPTY_NUDGE = "Your last reply was empty. Answer the question now."
FINAL_STEP_NOTE = (
    "You have no tool calls left. Answer now using only what the tools "
    "returned, and cite only papers they returned."
)

TOOL_FUNCTIONS = {
    "search_arxiv": search_arxiv,
    "get_abstract": get_abstract,
    "lookup_wikipedia": lookup_wikipedia,
}


@dataclass
class AgentResult:
    answer: str
    steps: int
    tracer: Tracer


def _run_tool(name: str, raw_args: str):
    func = TOOL_FUNCTIONS.get(name)
    if func is None:
        return f"There is no tool called {name}. Use one of: {', '.join(TOOL_FUNCTIONS)}."
    try:
        args = json.loads(raw_args or "{}")
        if not isinstance(args, dict):
            raise ValueError
    except ValueError:
        return "The tool arguments were not valid JSON. Try the call again."
    try:
        inspect.signature(func).bind(**args)
    except TypeError:
        return f"The arguments did not match {name}. Check the tool description."
    try:
        return func(**args)
    except Exception as error:
        return f"{name} failed ({type(error).__name__}). Try different input or another tool."


def _to_message_text(result) -> str:
    text = result if isinstance(result, str) else json.dumps(result)
    if len(text) > RESULT_CHARS:
        return text[:RESULT_CHARS] + TRUNCATION_MARK
    return text


def run_agent(
    client,
    question: str,
    system_prompt: str,
    tool_schemas: list[dict],
    max_steps: int = MAX_STEPS,
    tracer: Tracer | None = None,
    model: str = MODEL,
    extra_body: dict | None = None,
) -> AgentResult:
    """Run the agent until it answers or reaches ``max_steps`` model calls.

    Clears the tool caches first. A ``tool_use_failed`` error is sent
    back to the model once, and a second one ends the run with a message.
    """
    tracer = tracer or Tracer()
    reset_cache()
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    retried_tool_error = False
    retried_empty = False

    for step in range(1, max_steps + 1):
        start = time.monotonic()
        try:
            # The last call has tools off so a run that keeps searching still
            # ends with an answer built from what it already retrieved.
            last = step == max_steps and max_steps > 1
            response = chat(
                client,
                model=model,
                messages=messages + [{"role": "user", "content": FINAL_STEP_NOTE}] * last,
                tools=tool_schemas,
                extra_body=extra_body,
                **({"tool_choice": "none"} if last else {}),
            )
        except Exception as error:
            if "tool_use_failed" not in str(error):
                raise
            if retried_tool_error:
                text = "The model could not format a tool call, so the run stopped."
                tracer.answer(text)
                return AgentResult(text, step, tracer)
            retried_tool_error = True
            tracer.note(f"Step {step}: the model made a malformed tool call, retrying")
            messages.append(
                {
                    "role": "user",
                    "content": f"Your last tool call was malformed ({error}). "
                    "Call the tool again with valid arguments.",
                }
            )
            continue

        usage = getattr(response, "usage", None)
        tracer.model_call(
            step,
            _count(usage, "total_tokens"),
            time.monotonic() - start,
            prompt=_count(usage, "prompt_tokens"),
            completion=_count(usage, "completion_tokens"),
            cached=_count(getattr(usage, "prompt_tokens_details", None), "cached_tokens"),
            reasoning=_count(getattr(usage, "completion_tokens_details", None), "reasoning_tokens"),
            served_by=_served_by(response),
            cost=_cost(usage),
        )

        message = response.choices[0].message
        if not message.tool_calls:
            text = message.content or ""
            if not text.strip():
                if not retried_empty and step < max_steps:
                    retried_empty = True
                    tracer.note(f"Step {step}: the model sent an empty reply, asking again")
                    messages.append({"role": "user", "content": EMPTY_NUDGE})
                    continue
                text = EMPTY_ANSWER
            tracer.answer(text)
            return AgentResult(text, step, tracer)

        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": call.id,
                        "type": "function",
                        "function": {
                            "name": call.function.name,
                            "arguments": call.function.arguments,
                        },
                    }
                    for call in message.tool_calls
                ],
            }
        )
        for call in message.tool_calls:
            result = _run_tool(call.function.name, call.function.arguments)
            tracer.tool_call(step, call.function.name, _parsed_args(call), result)
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": _to_message_text(result),
                }
            )

    text = f"Stopped after {max_steps} steps without a final answer."
    tracer.answer(text)
    return AgentResult(text, max_steps, tracer)


def _cost(usage) -> float:
    value = getattr(usage, "cost", None)
    if value is None:
        value = (getattr(usage, "model_extra", None) or {}).get("cost")
    return float(value) if isinstance(value, (int, float)) else 0.0


def _served_by(response) -> str:
    name = getattr(response, "provider", None)
    if name is None:
        name = (getattr(response, "model_extra", None) or {}).get("provider")
    return name if isinstance(name, str) else ""


def _count(obj, name: str) -> int:
    value = getattr(obj, name, 0)
    return value if isinstance(value, int) else 0


def _parsed_args(call) -> dict:
    try:
        args = json.loads(call.function.arguments or "{}")
    except ValueError:
        return {}
    return args if isinstance(args, dict) else {}

In [ ]:
SYSTEM_PROMPT = (
    "You are a research paper assistant. Use search_arxiv to find papers, "
    "get_abstract to read one, and lookup_wikipedia to explain unfamiliar "
    "terms. Search arXiv at most twice, read at most two abstracts, then "
    "answer. arXiv search ranks by relevance and cannot filter by date, so "
    "do not put years in a query. Answer in a short paragraph and cite each "
    "paper by its arXiv id. Only cite papers your tools returned."
)

## 6. Ask a question

Watch the trace. Each step shows which tool the model chose and what came
back.

In [ ]:
QUESTION = (
    "Find a few papers on deep learning for protein structure prediction "
    "and explain in plain language what one of them does."
)

result = run_agent(client, QUESTION, SYSTEM_PROMPT, TOOLS)

## 7. Check the citations

Models sometimes invent papers. This check looks at every arXiv id and
quoted title in the answer and counts it only if a tool returned it in
this run. It checks that a cited paper was retrieved. It does not check
that what the answer says about the paper is true.

In [ ]:
report = check_citations(result.answer, result.tracer.tool_results)
print(report.summary)
if report.unverified:
    print("Not retrieved by any tool:", report.unverified)

## 8. A backup question

Same agent, a different topic.

In [ ]:
QUESTION = (
    "Find a few papers on machine learning for weather forecasting and "
    "explain in plain language what one of them does."
)

result = run_agent(client, QUESTION, SYSTEM_PROMPT, TOOLS)
report = check_citations(result.answer, result.tracer.tool_results)
print(report.summary)
if report.unverified:
    print("Not retrieved by any tool:", report.unverified)

## 9. Compare with a chatbot

Ask a chatbot with web search off the same first question. Then ask it
again with web search on. Look at which papers it names and whether you
could find each one. The difference is what the citation check is for.